In [1]:
#see README_WGS_analysis.txt for detailed instructions
import os
import re
import subprocess
import time
import pysam
notebook_path = os.getcwd()

#requirements: Staden Package (srf2fastq): https://sourceforge.net/projects/staden/
#Bowtie2 + hg19 index, Samtools

#Required arguments:

#Folder with patient SRF files to process:
folder_path = '/Volumes/4TBEncrypted/Downloads/Patients/PD3964a/'

os.chdir(folder_path)

#hg19 index location:
hg19idx = '/Volumes/4TBEncrypted/Downloads/hg19/hg19bt'
#number of CPU threads:
cpu_threads = '2'
#Amount of memory - note: appears to be PER thread for samtools; 
#make sure cpu_threads*memory is < than available RAM
memory = '10G'
samtools_mem = '2G'

#Add hg19 coordinates of Ig and TCR loci to a dictionary:

vdj = {}
vdj['kappa_locus'] = 'chr2:89,112,925-90,330,278'
vdj['lambda_locus'] = 'chr22:22,365,635-23,296,674'
vdj['igh_locus'] = 'chr14:106,015,294-107,319,860'
vdj['tcra_locus'] = 'chr14:22,090,057-23,021,075'
vdj['tcrb_locus'] = 'chr7:141,867,766-142,580,276'
vdj['tcrg_locus'] = 'chr7:38,219,396-38,478,150'
vdj['tcrd_locus'] = 'chr14:22,891,537-22,935,569'


def run_shell_command(command):
    print(f'Running command: {command} \n')
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, shell=True)
    output = process.communicate()
    print(output[1].decode("utf-8"))

# Step 1(a) SRF to FASTQ

In [ ]:
#Look in all subfolders for .srf files (might need to add extension manually to
#downloaded files from pyega3 client), run srf2fastq to extract forward and
#reverse fastq files

#Check for fastq files, add the file identifier to a list
fastq_files = []
for folder, sub_folders, files in os.walk(os.getcwd()):
    for fastq_file in files:
        match = re.search(r'EGAF(\w+)_forward.fastq$', fastq_file)
        if match:
            fastq_files.append(match.group().split('_')[0])

#Look for srf files, run srf2fastq IF fastq not already present
for folder, sub_folders, files in os.walk(os.getcwd()):
    for srf_file in files:
        match = re.search(r'EGAF(\w+).srf$', srf_file)
        if match:
            if match.group().split('.')[0] in fastq_files:
                #If file ID already has a fastq file, do nothing
                pass
            else:
                command = 'srf2fastq -c -s {} {}'.format(folder+'/'+os.path.splitext(srf_file)[0],folder+'/'+srf_file)
                run_shell_command(command)

# Step 1(b) BAM to FASTQ

In [ ]:
#Look in all subfolders for .bam files (might need to add extension manually to
#downloaded files from pyega3 client), run srf2fastq to extract forward and
#reverse fastq files

#Check for fastq files, add the file identifier to a list
fastq_files = []
for folder, sub_folders, files in os.walk(os.getcwd()):
    for fastq_file in files:
        match = re.search(r'EGAF(\w+)_forward.fastq', fastq_file)
        if match:
            fastq_files.append(match.group().split('_')[0])

#Look for bam files, sort with samtools and use bedtools to extract fastq IF fastq not already present
for folder, sub_folders, files in os.walk(os.getcwd()):
    for bam_file in files:
        match = re.search(r'EGAF(\w+).bam$', bam_file)
        if match:
            if match.group().split('.')[0] in fastq_files:
                #If file ID already has a fastq file, do nothing
                pass
            else:
                #sort with samtools
                bam_input = arg2 = folder+'/'+bam_file
                sorted_bam_output = folder+'/'+os.path.splitext(bam_file)[0]+'.qsort.bam'
                
                command = f'samtools sort -m {samtools_mem} -@ {cpu_threads} -n {bam_input} -o {sorted_bam_output}'
                run_shell_command(command)
                
                #bamtofastq
                fastq_output = folder+'/'+os.path.splitext(bam_file)[0]
                
                command = f'samtools fastq -@ {cpu_threads} -1 {fastq_output}_forward.fastq.gz -2 {fastq_output}_reverse.fastq.gz -0 /dev/null -s /dev/null -n {sorted_bam_output} '
                run_shell_command(command)
                
                #delete sorted bam to save space
                command = f'rm {sorted_bam_output}'
                run_shell_command(command)
                

# Step 2: Align FASTQ to human genome (hg19 build) using Bowtie2

In [ ]:
#Look in all subfolders for fastq files (x_foward.fastq and x_reverse.fastq)
#Just search for .forward.fastq, and use fastq_file.split() to get file name
#identifier without _forward/_reverse.

#Check for _bt2.sam files, add the file identifier to a list
bam_files = []t

for folder, sub_folders, files in os.walk(os.getcwd()):
    for bam_file in files:
        match = re.search(r'EGAF(\w+)_bt2.bam$', bam_file)
        if match:
            bam_files.append(match.group().split('_')[0])

for folder , sub_folders , files in os.walk(os.getcwd()):
    
    for fastq_file in files:
        match = re.search(r'EGAF(\w+)_forward.fastq$',fastq_file)
        match_gz = re.search(r'EGAF(\w+)_forward.fastq.gz$',fastq_file)
        
        if match:
            if match.group().split('_')[0] in bam_files:
                #If file ID already has a _bt2.sam file, do nothing
                pass
            else:
                fastq_f = folder+'/'+fastq_file.split('_')[0]+'_forward.fastq'
                fastq_r = folder+'/'+fastq_file.split('_')[0]+'_reverse.fastq'
                bam_output = folder+'/'+fastq_file.split('_')[0]+'_bt2.bam'

                command = f'bowtie2 --local -x {hg19idx} -p {cpu_threads} -1 {fastq_f} -2 {fastq_r} | samtools view -bS - > {bam_output}'
                run_shell_command(command)
        
        if match_gz:
            if match_gz.group().split('_')[0] in bam_files:
                #If file ID already has a _bt2.sam file, do nothing
                pass
            else:
                fastq_f = folder+'/'+fastq_file.split('_')[0]+'_forward.fastq.gz'
                fastq_r = folder+'/'+fastq_file.split('_')[0]+'_reverse.fastq.gz'
                bam_output = folder+'/'+fastq_file.split('_')[0]+'_bt2.bam'

                command = f'bowtie2 --local -x {hg19idx} -p {cpu_threads} -1 {fastq_f} -2 {fastq_r} | samtools view -bS - > {bam_output}'
                run_shell_command(command)
                
                

# Step 3: Samtools:

In [ ]:
#Check if sorted bam already present
bam_files = []

for folder, sub_folders, files in os.walk(os.getcwd()):
    for bam_file in files:
        match = re.search(r'EGAF(\w+)_bt2.sorted.bam$', bam_file)
        if match:
            bam_files.append(match.group().split('_')[0])

#Sort bam
for folder, sub_folders, files in os.walk(os.getcwd()):
    for bam_file in files:
        match = re.search(r'EGAF(\w+)_bt2.bam$', bam_file)
        if match:
            if match.group().split('_')[0] in bam_files:
                #If file ID already has a _bt2.sorted.bam file, do nothing
                pass
            else:
                bam_input = folder+'/'+bam_file
                bam_output = folder+'/'+bam_file.split('.')[0]+'.sorted.bam'

                #Sort bam
                command = f'samtools sort -m {samtools_mem} -@ {cpu_threads} {bam_input} -o {bam_output}'
                run_shell_command(command)

                #Index bam
                command = f'samtools index -@ {cpu_threads} -b {bam_output}'
                run_shell_command(command)

# Step 4 - Extract reads from IG and TCR loci

In [ ]:
# Extract VDJ loci from sorted bam files
for folder, sub_folders, files in os.walk(os.getcwd()):
    for bam_file in files:
        match = re.search(r'EGAF(\w+)_bt2.sorted.bam$', bam_file)
        if match:
            bam_input = folder+'/'+bam_file.split('.')[0]+'.sorted.bam'
            for key in vdj:
                bam_output = folder+'/'+bam_file.split('.')[0]+'.sorted.'+key+'.bam'
                command = f'samtools view -h {bam_input} "{vdj[key]}" > {bam_output}'
                run_shell_command(command)

# Merge bam files for each locus for patient
for locus in vdj:
    locus_list = []
    for folder, sub_folders, files in os.walk(os.getcwd()):
    
        for bam_file in files:
            match = re.search(fr'EGAF\w+_bt2.sorted.{locus}.bam$', bam_file)
            if match:
                locus_list.append(folder+'/'+bam_file.split('.')[0]+'.sorted.'+locus+'.bam')
    
    locus_files = (' ').join(locus_list)
    #Folder name is patient ID
    patient_identifier = os.getcwd().split('/')[-1]
    output_file = f'{os.getcwd()}/{patient_identifier}_{locus}'
    
    command = f'samtools merge {output_file}.bam {locus_files}'
    run_shell_command(command)
    command = f'samtools sort {output_file}.bam -o {output_file}.sorted.bam'
    run_shell_command(command)
    command = f'samtools index {output_file}.sorted.bam'
    run_shell_command(command)